In [1]:
import pandas as pd
import numpy as np
from modulos.data_load import load, coins
from sklearn.preprocessing import MinMaxScaler

In [ ]:

resultados = {}

for coin in coins:
    df = load(coin)
    print(f"{coin} →", df.columns.tolist())
    print(f'Processando moeda: {coin}')

    df = df.sort_values('date').reset_index(drop=True)
    
    # Features derivadas
    ## Preço
    df['faixa_preco'] = df['high'] - df['low']
    df['retorno_pct_7d'] = df['close'].pct_change(7)
    df['momentum_7d'] = df['close'] - df['close'].shift(7)

    ## Indicadores estatísticos
    df['media_movel_7d'] = df['close'].rolling(window=7).mean()
    df['std_7d'] = df['close'].rolling(window=7).std()

    ## Liquidez
    df['volume_2_7d'] = df['Volume 2'].rolling(window=7).mean()
    df['taker_ratio'] = df['buyTakerAmount'] / df['Volume 2']
    df['buy_pressure'] = df['buyTakerQuantity'] / df['tradeCount']
    df['volume_volatilidade_ratio'] = df['Volume 2'] / df['faixa_preco']


    df['date'] = pd.to_datetime(df['date'])
    df['dia_da_semana'] = df['date'].dt.dayofweek
    
    # Limpando NaNs
    df.dropna(inplace=True)
    
    # Separa os conjuntos
    n = len(df)
    n_train = int(n * 0.7)
    n_val = int(n * 0.15)

    df_train = df.iloc[:n_train]
    df_val = df.iloc[n_train:n_train + n_val]
    df_test = df.iloc[n_train + n_val:]

    # Normaliza (somente com treino!)
    features = [
        'close', 'media_movel_7d',
        'std_7d', 'momentum_7d', 'retorno_pct_7d',
        'volume_2_7d', 'taker_ratio', 'buy_pressure', 'volume_volatilidade_ratio',
        'dia_da_semana'
    ]
    
    scaler = MinMaxScaler()
    scaler.fit(df_train[features])

    df_train[features] = scaler.transform(df_train[features])
    df_val[features] = scaler.transform(df_val[features])
    df_test[features] = scaler.transform(df_test[features])
    
    # Salva para uso posterior (opcional)
    resultados[coin] = {
        'train': df_train,
        'val': df_val,
        'test': df_test,
        'scaler': scaler
    }

    print(f'{coin} - Train shape: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}')


AAVEBTC → ['unix', 'date', 'symbol', 'open', 'high', 'low', 'close', 'Volume 1', 'Volume 2', 'buyTakerAmount', 'buyTakerQuantity', 'tradeCount', 'weightedAverage']
Processando moeda: AAVEBTC


NameError: name 'df_moeda' is not defined